![image.png](https://i.imgur.com/a3uAqnb.png)

# Pretraining a language model from scratch
in this lab we'll look at pretraining a langauge model, you've seen this before in the SLL labs, where we train a model on a large amount of unlabled data then fine tune it on the limited amount of labled data. NLP really pushed pretraining far.

in this lab we will: Take a real book → clean it → learn a vocabulary → initialise a model → train it until it writes like the book.

Then poke the result until it tells us what it did and did not learn.

The book we'll train on is G. H. Hardy's **A Course of Pure Mathematics** (1908), https://archive.org/details/coursepuremath00hardrich/page/n19/mode/2up

it's available from Project
Gutenberg only as LaTeX source. but That is the first lesson: data doesn't usually arive clean.



LaTeX is basically a language that, when compiled, produces a PDF. It is treated in academia as almost a necessity because it allows you to write complicated math in a clean form on a page...For example:$$\int_{0}^{\infty} \phi(x)\, dx = \sum_{n=0}^{\infty} \frac{1}{(n + 1)^{2}}$$...is actually just this raw string compiled:\int_{0}^{\infty} \phi(x)\, dx = \sum_{n=0}^{\infty} \frac{1}{(n + 1)^{2}}

---
## 1. Data

Download the LaTeX source and strip it to readable text. Every regex below exists because
something broke without it: editorial macros (`\DPtypo`, `\Chg`), page-index sludge
(`\PgNo`), structural markup to delete, unwrap, or keep. The `assert` at the bottom is a
canary — reorder the substitutions and an earlier pattern eats the math commands.

In [ ]:
import re, requests

URL = "https://www.gutenberg.org/files/38769/38769-t/38769-t.tex"
tex = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=120).text

m = re.search(r"\\begin\{document\}(.*)\\end\{document\}", tex, re.S)
body = m.group(1) if m else tex
body = re.sub(r"(?<!\\)%.*", "", body)

# start at Chapter I — drops title page, ToC, and the \PgNo index sludge
c = re.search(r"\\Chapter\{I\}", body)
if c: body = body[c.start():]

# PG editorial macros: keep the corrected reading, drop the original
body = re.sub(r"\\(DPchg|DPtypo|Chg)\{[^{}]*\}\{([^{}]*)\}", r"\2", body)
body = re.sub(r"\\(Add|Del)\{([^{}]*)\}", r"\2", body)
body = re.sub(r"\\SecNo\[[^\]]*\]\{([^{}]*)\}", r"§\1", body)
body = re.sub(r"\\(ie|eg|etc)\\?(?![a-zA-Z])", "", body)
body = re.sub(r"\\sqrtb?r?\{", r"\\sqrt{", body)
body = re.sub(r"\\Chapter\{[^{}]*\}\{([^{}]*)\}", r"\n\n\1\n", body)

DROP = ["PgNo","ToCPar","ToCApp","ToCBox","label","index","hypertarget",
        "hyperlink","phantomsection","includegraphics","PageSep","Pagelabel",
        "tnpage","BookMark","iffalse"]
KEEP = ["Paragraph","Section","Tag","Eq","Ex","Item","emph","textit","textbf",
        "textrm","textsc","text","mbox","footnotetext","footnote"]

for cmd in DROP:
    body = re.sub(r"\\" + cmd + r"\*?\s*(\[[^\]]*\])?(\{[^{}]*\})*", "", body)
for cmd in KEEP:
    for _ in range(3):
        body = re.sub(r"\\" + cmd + r"\*?\{([^{}]*)\}", r"\1", body)

body = re.sub(r"\\(begin|end)\{(itemize|enumerate|center|quote|small|Examples|"
              r"footnotesize|figure|table|Remark|Example)\*?\}(\{[^{}]*\})?", "", body)
body = body.replace(r"\item", "\n-")
body = re.sub(r"[ \t]+", " ", body)
body = re.sub(r"\n\s*\n\s*\n+", "\n\n", body).strip()

open("hardy.txt", "w", encoding="utf-8").write(body)

print(len(body), "chars |", len(body.split()), "words")

### What the cleaning did

Left: raw LaTeX. Right: what the model will see.

In [ ]:
from IPython.display import HTML, display
import html as _html

def side_by_side(left, right, left_title="RAW LaTeX", right_title="CLEANED"):
    display(HTML(
        "<table style='width:100%;table-layout:fixed;border-collapse:collapse'>"
        f"<tr><th style='text-align:left;padding:4px'>{left_title}</th>"
        f"<th style='text-align:left;padding:4px'>{right_title}</th></tr>"
        "<tr>"
        f"<td style='vertical-align:top;padding:6px;border:1px solid #ccc'><pre style='white-space:pre-wrap;font-size:11px;margin:0'>{_html.escape(left)}</pre></td>"
        f"<td style='vertical-align:top;padding:6px;border:1px solid #ccc'><pre style='white-space:pre-wrap;font-size:11px;margin:0'>{_html.escape(right)}</pre></td>"
        "</tr></table>"))

i = tex.find(r"\Chapter{I}")
side_by_side(tex[i:i+900], body[:900])

# and a passage with real mathematics in it
j_raw   = tex.find(r"\lim", i)
j_clean = body.find(r"\lim")
side_by_side(tex[j_raw-350:j_raw+550], body[j_clean-350:j_clean+550],
             "RAW LaTeX (math)", "CLEANED (math)")

some stuff still survived tho ... `$\lim_{x \to
\infty}$`, which is good ! it's the actual Math in LaTeX, our model should predict the math in LaTeX which is easier to orginise then unicode.


in reality, most work almost always lies on data then work on the model architecture

### Configuration
Every hyperparameter lives here

In [ ]:
import importlib.util, subprocess, sys
for pkg in ("transformers", "tokenizers"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os, math, json, random, time, contextlib
import numpy as np
import torch
import torch.nn.functional as F
from transformers import GPT2Config, GPT2LMHeadModel, PreTrainedTokenizerFast

# ---------------------------------------------------------------- reproducibility
SEED = 1337

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = (DEVICE == "cuda")

# ---------------------------------------------------------------- tokenizer
VOCAB_SIZE     = 2048
TOKENIZER_PATH = "bpe-2048.json"

# ---------------------------------------------------------------- model
N_POSITIONS = 512        # maximum context the model can ever attend over
N_EMBD      = 384
N_LAYER     = 6
N_HEAD      = 6

# ---------------------------------------------------------------- pretraining
BLOCK_SIZE  = 256        # training window (<= N_POSITIONS)
BATCH_SIZE  = 32
MAX_STEPS   = 3000
LR          = 3e-4
MIN_LR      = 3e-5
WARMUP      = 100
WEIGHT_DECAY = 0.1
GRAD_CLIP   = 1.0
EVAL_EVERY  = 300
EVAL_ITERS  = 40
VAL_FRAC    = 0.10

# ---------------------------------------------------------------- checkpoint
CKPT_DIR = "ckpt_pretrained"

# ---------------------------------------------------------------- sampling
# GEN_SHORT is used for the step-0 / mid / end comparison, so all three are identical
# in everything except how much training has happened.
GEN_SHORT = dict(max_new_tokens=120, temperature=0.9, top_p=0.95, do_sample=True)
GEN_PAGE  = dict(max_new_tokens=480, temperature=0.9, top_p=0.95, do_sample=True)

---
## 2. Tokenizer

We'll use GPT-2's model ... but not it's tokenizer

**Why not GPT-2's tokenizer?** It has 50,257 entries. Our embedding table is one row of
width `n_embd` per entry:

```
50,257 × 384 = 19.3M params   ← embeddings alone
       vs 10.6M               ← the entire rest of the transformer
```

On a corpus of ~175k words, most of those rows would be touched a handful of times and
leave training as the noise they started as. With `vocab_size=2048` trained on *this*
corpus, embeddings are 0.79M — a small slice, and every row earns its place.

We'll use BPE tokenizer, basicly words like "play" and "played" and "plays" will occupy 3 tokens usually when play is in all of them, so it makes "play" a token and "ed" as a different token,
simply put it groups commenly accuring segments into tokens so it's very efficient.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders


tok = Tokenizer(models.BPE(unk_token=None))

# Byte-level: every possible byte is representable, so nothing is ever <unk>.
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder       = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["<|endoftext|>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),  # the 256 byte symbols
    show_progress=True,
)

t0 = time.time()
tok.train(["hardy.txt"], trainer)
tok.save(TOKENIZER_PATH)
print(f"trained in {time.time()-t0:.1f}s  |  vocab size: {tok.get_vocab_size()}")

# Wrap in the HF interface so model.generate() can consume it directly.
hf_tok = PreTrainedTokenizerFast(
    tokenizer_object=tok,
    eos_token="<|endoftext|>",
    pad_token="<|endoftext|>",
    model_max_length=N_POSITIONS,
)
print("eos id:", hf_tok.eos_token_id)

### What BPE learned

BPE starts with 256 single-byte tokens and repeatedly merges the most frequent adjacent
pair. The list is in learning order, so it reads as a record of what this corpus is made
of: generic English first, Hardy-specific later.

(`Ġ` = space, `Ċ` = newline, byte-level encoded.)

### Where the boundaries fall
claude cooked visualizations

Each coloured block is one token. Common words are single tokens; rare words shatter.

In [ ]:
import html
from IPython.display import HTML, display

# visualization code
def show_tokens(text, title=""):
    enc = tok.encode(text)
    # Modern pastel palette with high contrast dark text
    palette = ["#e0f2fe", "#fef3c7", "#dcfce7", "#fee2e2", "#f3e8ff", "#ffedd5"]
    spans = []
    for k, t in enumerate(enc.tokens):
        # Escape special characters and convert GPT BPE whitespace indicators
        vis = html.escape(t.replace("Ġ", "␣").replace("Ċ", "⏎"))
        bg = palette[k % len(palette)]

        # Crisp pill-style tokens with hover highlight
        spans.append(
            f'<span style="background:{bg}; color:#0f172a; padding:2px 5px; '
            f'margin:2px 1px; border-radius:4px; font-family:\'Fira Code\', monospace; '
            f'font-size:13px; border:1px solid rgba(0,0,0,0.08); display:inline-block; '
            f'line-height:1.2" title="Token ID: {enc.ids[k]}">{vis}</span>'
        )
    # Render header and token sequence container
    display(HTML(
        f'<div style="font-family:-apple-system,BlinkMacSystemFont,sans-serif; margin-bottom:12px; '
        f'padding:12px; border-radius:8px; background:rgba(128,128,128,0.05); border:1px solid rgba(128,128,128,0.2)">'
        f'  <div style="font-size:14px; margin-bottom:8px">'
        f'    <b style="color:#2563eb">{html.escape(title)}</b> '
        f'    <span style="opacity:0.6; font-size:12px">• {len(text)} chars → <b>{len(enc.ids)} tokens</b></span>'
        f'  </div>'
        f'  <div style="line-height:2.0; display:flex; flex-wrap:wrap; gap:1px">{"".join(spans)}</div>'
        f'</div>'
    ))
    return enc

# --- Execution ---
snippet = body[body.find(r"\lim") - 120: body.find(r"\lim") + 180]
enc = show_tokens(snippet, "a LaTeX passage from Hardy")

print("\nfirst 12 token ids :", enc.ids[:12])
print("first 12 as strings:", enc.tokens[:12])
print()

---
## 3. Dataset

Encode the book into one long 1-D tensor, hold out the last 10%. Split by position, not
by shuffling — shuffling would leak the end of the book into training through overlapping
windows.

In [ ]:
raw = open("hardy.txt", encoding="utf-8").read()
ids = tok.encode(raw).ids
data = torch.tensor(ids, dtype=torch.long)

n_val   = int(len(data) * VAL_FRAC)
n_train = len(data) - n_val
train_data, val_data = data[:n_train], data[n_train:]

print(f"corpus      : {len(raw):,} characters")
print(f"tokenised   : {len(data):,} tokens")
print(f"train / val : {len(train_data):,} / {len(val_data):,}")
print(f"one epoch   = {len(train_data) / (BATCH_SIZE * BLOCK_SIZE):.1f} steps at batch {BATCH_SIZE} x block {BLOCK_SIZE}")
print(f"planned run = {MAX_STEPS} steps = {MAX_STEPS * BATCH_SIZE * BLOCK_SIZE / len(train_data):.1f} epochs")
print(f"\nfirst 30 tokens: {data[:30].tolist()}")
print(f"decoded back   : {tok.decode(data[:30].tolist())!r}")

In [ ]:
def get_batch(split="train", batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, source=None):
    d = source if source is not None else (train_data if split == "train" else val_data)
    # one random starting offset per sequence in the batch
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i     : i + block_size    ] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])   # <- shifted by ONE
    return x.to(DEVICE), y.to(DEVICE)

xb, yb = get_batch("train")
print("x:", tuple(xb.shape), " y:", tuple(yb.shape))

---
## 4. The model

`GPT2LMHeadModel` — but from a config we write, not `from_pretrained`.

> **There are no pretrained weights in this notebook.** `GPT2LMHeadModel(config)`
> allocates tensors and fills them with normal noise. We borrow the *architecture* from
> Hugging Face (attention wiring, KV-cached `.generate()`)

In [ ]:
set_seed()

config = GPT2Config(
    vocab_size=VOCAB_SIZE,
    n_positions=N_POSITIONS,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    bos_token_id=hf_tok.eos_token_id,
    eos_token_id=hf_tok.eos_token_id,
)

model = GPT2LMHeadModel(config).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(model.config.model_type, "| randomly initialised, no pretrained weights loaded")
print(f"parameters: {n_params:,}  ({n_params/1e6:.2f}M)")

In [ ]:
from collections import OrderedDict

groups = OrderedDict((k, 0) for k in
    ["token embedding (wte)", "position embedding (wpe)", "attention", "mlp", "layernorm"])

for name, p in model.named_parameters():
    if   name.startswith("transformer.wte"): groups["token embedding (wte)"] += p.numel()
    elif name.startswith("transformer.wpe"): groups["position embedding (wpe)"] += p.numel()
    elif ".attn." in name:                   groups["attention"] += p.numel()
    elif ".mlp."  in name:                   groups["mlp"] += p.numel()
    else:                                    groups["layernorm"] += p.numel()

total = sum(groups.values())
print(f"{'component':<26}{'params':>12}{'share':>9}")
print("-" * 47)
for k, v in groups.items():
    print(f"{k:<26}{v:>12,}{v/total*100:>8.1f}%")
print("-" * 47)
print(f"{'TOTAL':<26}{total:>12,}{100.0:>8.1f}%")

transformer_only = total - groups["token embedding (wte)"] - groups["position embedding (wpe)"]
print(f"\n(the output head shares its weights with wte — tied, so counted once)")

print("\n" + "=" * 60)
print("THE COUNTERFACTUAL: same model, GPT-2's tokenizer")
print("=" * 60)
gpt2_emb = 50257 * N_EMBD
ours_emb = VOCAB_SIZE * N_EMBD
print(f"transformer blocks (unchanged) : {transformer_only:>12,}")
print(f"embeddings with vocab  2,048   : {ours_emb:>12,}   ({ours_emb/(transformer_only+ours_emb)*100:.1f}% of model)")
print(f"embeddings with vocab 50,257   : {gpt2_emb:>12,}   ({gpt2_emb/(transformer_only+gpt2_emb)*100:.1f}% of model)")

---
## 5. Pretraining

A manual loop. Six lines learn: draw batch → forward → cross-entropy against the shifted
targets → backward → clip and step → zero grads. Everything else is instrumentation.

- **AdamW, betas (0.9, 0.95), wd 0.1** — standard GPT recipe; no decay on biases/LayerNorm.
- **Warmup then cosine decay** — step 0 gradients are large and uninformative.
- **Clip at 1.0** — insurance against one pathological batch.
- **Mixed precision** — ~2× faster on a T4; `GradScaler` keeps small grads from
  underflowing fp16.

In [ ]:
@torch.no_grad()
def estimate_loss(m, iters=EVAL_ITERS, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE,
                  sources=None):
    # average loss over a few random batches, for each provided data source
    m.eval()
    out = {}
    splits = sources if sources is not None else {"train": train_data, "val": val_data}
    for name, src in splits.items():
        losses = torch.zeros(iters)
        for k in range(iters):
            x, y = get_batch(batch_size=batch_size, block_size=block_size, source=src)
            with amp_ctx:
                logits = m(x).logits
                losses[k] = F.cross_entropy(logits.view(-1, logits.size(-1)), y.reshape(-1)).item()
        out[name] = losses.mean().item()
    m.train()
    return out


@torch.no_grad()
def sample(prompt="\n", seed=SEED, **gen_kwargs):
    # sample with HF .generate(); seeded, so runs are reproducible
    model.eval()
    set_seed(seed)
    enc = hf_tok(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(**enc, pad_token_id=hf_tok.eos_token_id, **gen_kwargs)
    model.train()
    return hf_tok.decode(out[0], skip_special_tokens=True)


amp_ctx = torch.autocast("cuda", dtype=torch.float16) if USE_AMP else contextlib.nullcontext()

In [ ]:
# sample from untrained model !
samples = {}
samples["step 0 (random weights)"] = sample("Proving that", **GEN_SHORT)
print(samples["step 0 (random weights)"])

Uniform noise over the vocabulary — textual static. Now train; watch the two loss columns.

In [ ]:
set_seed()
model.train()

# no weight decay on 1-D parameters (biases, LayerNorm gains)
decay     = [p for n, p in model.named_parameters() if p.dim() >= 2]
no_decay  = [p for n, p in model.named_parameters() if p.dim() <  2]
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=LR, betas=(0.9, 0.95))

try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def lr_at(step):
    if step < WARMUP:                                   # linear warmup
        return LR * (step + 1) / WARMUP
    prog = (step - WARMUP) / max(1, MAX_STEPS - WARMUP)  # cosine decay
    return MIN_LR + 0.5 * (LR - MIN_LR) * (1 + math.cos(math.pi * prog))

history, t0 = [], time.time()
MID = MAX_STEPS // 2

for step in range(MAX_STEPS):

    lr = lr_at(step)
    for g in optimizer.param_groups:
        g["lr"] = lr

    # 1. batch
    x, y = get_batch("train")

    # 2-3. forward + loss (we shift ourselves; see the aside in §3)
    with amp_ctx:
        logits = model(x).logits
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.reshape(-1))

    # 4-6. backward, clip, step
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()

    if step % EVAL_EVERY == 0 or step == MAX_STEPS - 1:
        L = estimate_loss(model)
        history.append((step, L["train"], L["val"]))
        print(f"step {step:>5} | train {L['train']:.4f} | val {L['val']:.4f} "
              f"| lr {lr:.2e} | {time.time()-t0:>5.0f}s")

    if step == MID:
        samples[f"step {MID} (mid-training)"] = sample("Proving that", **GEN_SHORT)

print(f"\ndone in {time.time()-t0:.0f}s")

In [ ]:
import matplotlib.pyplot as plt

steps  = [h[0] for h in history]
plt.figure(figsize=(7, 4))
plt.plot(steps, [h[1] for h in history], label="train", marker="o", ms=3)
plt.plot(steps, [h[2] for h in history], label="val",   marker="o", ms=3)
plt.axhline(math.log(VOCAB_SIZE), ls="--", c="grey", lw=1, label=f"random guess ln({VOCAB_SIZE})")
plt.xlabel("step"); plt.ylabel("cross-entropy loss")
plt.title("Pretraining"); plt.legend(); plt.grid(alpha=.3)
plt.show()

### The three samples

Same prompt, same sampling settings, same seed. The only variable is training.

In [ ]:
samples[f"step {MAX_STEPS} (end of pretraining)"] = sample("Proving that", **GEN_SHORT)
for title, text in samples.items():
    print("=" * 78)
    print(f"  {title}")
    print("=" * 78)
    print(text.strip()[:700])
    print()

Step 0: noise. Mid: word-*shaped* things — right spacing, right capitalisation, `$` in
plausible places, no meaning. End: recognisably Hardy.

None of it was designed. No grammar module, no LaTeX parser, no spelling rule it all
fell out of minimising cross-entropy, because to predict text well you must internalise
spelling, then morphology, then syntax, then register.

If val loss flattens or turns up while train loss falls, that is memorisation: 11M
parameters against a few hundred thousand tokens is a generous ratio. Real runs sit on the
other side of that line.

In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)
model.save_pretrained(CKPT_DIR)
hf_tok.save_pretrained(CKPT_DIR)
print("saved to", CKPT_DIR, "->", sorted(os.listdir(CKPT_DIR)))

---
## 6. A page of Hardy

Ask for ~1500 characters of continuous text.

In [ ]:
page = sample("\n", seed=SEED, **GEN_PAGE)
page = page.strip()
print(f"[{len(page)} characters]\n")
print(page)

### Syntax without semantics

**The syntax is right.** Braces nest. `\frac` takes two groups, `\lim`
takes a subscript. Sentences have subjects and verbs, the register is consistent, it hedges
like a 1908 textbook. Much of it would compile.

**The mathematics is meaningless.** The theorems are not theorems, the limits do not
converge to what it claims, nothing follows from the line above.

---
## 7. Playground

The model is trained. Now prod it. Everything below runs in seconds, so change the
prompts and re-run.

### 7.1 The temperature dial

One prompt, one seed, five temperatures. Low = the model's favourite continuation, over
and over. High = it starts sampling from the tail and falls apart.

In [ ]:
PROMPT = "\n\nTHEOREM. If $f(x)$ is continuous"

for temp in [0.2, 0.5, 0.8, 1.0, 1.4]:
    out = sample(PROMPT, seed=SEED, max_new_tokens=60,
                 temperature=temp, top_p=1.0, do_sample=True)
    print("=" * 78)
    print(f"  temperature = {temp}")
    print("=" * 78)
    print(out.strip()[:400], "\n")

### 7.2 What does it think comes next?

Greedy decoding is the model's *mode*. Here is the actual distribution behind the first
step — the top 12 candidates, with their probabilities, for a few contexts.

In [ ]:
def piece_of(i):
    """token string for an id (the model's vocab can be a hair wider than the tokenizer's)"""
    return tok.id_to_token(i) or f"<{i}>"

# claude cooked visualizations
@torch.no_grad()
def next_tokens(prompt, k=12):
    model.eval()
    ids = hf_tok(prompt, return_tensors="pt").to(DEVICE)
    probs = F.softmax(model(**ids).logits[0, -1].float(), dim=-1)
    p, i = probs.topk(k)
    rows = ""
    for prob, idx in zip(p.tolist(), i.tolist()):
        piece = piece_of(idx).replace("Ġ", "␣").replace("Ċ", "⏎")
        rows += (
            f"<tr><td style='font-family:monospace;padding:2px 8px'>{_html.escape(piece)}</td>"
            f"<td style='padding:2px 8px;width:60%'>"
            f"<div style='background:#60a5fa;height:12px;width:{prob*100:.1f}%;"
            f"min-width:1px;border-radius:2px'></div></td>"
            f"<td style='padding:2px 8px;font-family:monospace'>{prob:.3f}</td></tr>")
    display(HTML(
        f"<div style='margin:8px 0 2px'><b>after</b> "
        f"<code>{_html.escape(prompt[-60:])}</code></div>"
        f"<table style='border-collapse:collapse;width:520px'>{rows}</table>"))
    model.train()

next_tokens("The function $f(x)$ is")
next_tokens("$\\lim_{x \\to")
next_tokens("\\frac{1}{")
next_tokens("It follows that")

Look at the LaTeX ones. The model has learned that `\frac{1}{` is essentially a promise:
almost all the mass sits on a handful of tokens. That is syntax as probability mass.

### 7.5 Is it writing or remembering?

The uncomfortable question. Take the generated page and find the longest run of words that
appears verbatim in the training corpus.

In [ ]:
corpus_words = " " + " ".join(body.split()) + " "
gen_words = page.split()

longest, longest_seq = 0, ""
for n in range(3, 41):
    hit = None
    for i in range(len(gen_words) - n + 1):
        s = " ".join(gen_words[i:i + n])
        if " " + s + " " in corpus_words:
            hit = s
            break
    if hit is None:
        break
    longest, longest_seq = n, hit

print(f"generated page: {len(gen_words)} words")
print(f"longest verbatim run found in the corpus: {longest} words\n")
if longest:
    print(f"  ...{longest_seq}...\n")
else:
    print("  (not even a 3-word run — everything here is freshly assembled)\n")
print("Short runs are inevitable — any fluent English shares 4-grams with any other.")
print("Long runs would mean the model is reciting rather than composing.")

### 7.7 Who is attending to whom?

Attention weights from the last block: row = the token doing the looking, column = what it
looks at.

In [ ]:
import matplotlib.pyplot as plt

def set_attn_impl(impl):
    """attention weights are only exposed by the eager kernel, not sdpa/flash"""
    try:
        model.set_attn_implementation(impl)        # newer transformers
    except Exception:
        model.config._attn_implementation = impl   # older transformers
# claude cooked visualization
@torch.no_grad()
def attention_map(text, layer=-1, head=0):
    model.eval()
    prev = getattr(model.config, "_attn_implementation", "eager")
    set_attn_impl("eager")
    try:
        ids = torch.tensor(tok.encode(text).ids[:36], device=DEVICE)[None]
        atts = model(ids, output_attentions=True).attentions
        if not atts:
            print("this transformers build will not hand back attention weights")
            return
        att = atts[layer][0, head].float().cpu()
    finally:
        set_attn_impl(prev)

    labels = [piece_of(t).replace("Ġ", "␣").replace("Ċ", "⏎")
              for t in ids[0].tolist()]

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(att, cmap="magma")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=7)
    ax.set_title(f"layer {layer}, head {head}")
    ax.set_xlabel("attended to"); ax.set_ylabel("attending from")
    plt.tight_layout(); plt.show()
    model.train()
# how come the upper triangle is empty ?
attention_map("If $f(x)$ is continuous for $x = a$, then $\\lim_{x \\to a} f(x) = f(a)$.")

### 7.8 Feed it something it has never seen

The corpus is one 1908 maths textbook. Hand it a prompt from outside that world and watch
it drag everything back to Hardy — the only thing it can do is continue text in the one
register it knows.

In [ ]:
alien = [
    "\n\nDear Sir, I am writing to complain about",
    "\n\nRECIPE. To make a good soup, first",
    "\n\ndef train(model, data):",
    "\n\nOnce upon a time there was a dragon who",
]

for p in alien:
    out = sample(p, seed=SEED, max_new_tokens=70, temperature=0.8,
                 top_p=0.95, do_sample=True)
    print("-" * 78)
    print("PROMPT:", repr(p.strip()))
    print(out[len(p):].strip()[:320])
    print()

---
## 8. Takeaway

Our model was close enough `for this scale`. The thing that differentiates this pretraining
from GPT-3's pretraining is scaling more data and a bigger model, and it starts spitting
out actually coherent math, in its own style.

One book is essentially nothing, even if it's da goat G. H. Hardy. A 10M-parameter model
overfits on it easily.

- **The way the model talks is cheap. The knowledge it has is expensive.**

Scaled up, you could fine-tune it on chatting and it would perform well, chatting about
things it never encountered in the fine-tuning data, keeping its style and drawing on the
knowledge it gained in pretraining. Depending on the scale.

To make it "understand" maths (a big discussion in the AI field) you'd need reasoning and RL.

Things to try: KEEP SCALING UP AND SEE WHAT HAPPENS ! (just like **literally** everyone ever - **AI** is pay to win type shi)

### Lab Cooked by Muhannad

<img src="https://i.imgur.com/erzOsMx.jpeg" width="800">